# 🛠️ Notebook 2: Parking Lot — Bad → Good → Best

This notebook shows **three versions** of the same feature and explains *why each
step is an improvement*. Each version is fully runnable so you can compare them.

| Version | Style | Problem it fixes |
|---------|-------|------------------|
| 🚫 Bad  | "God class" — one big class does everything | Baseline — shows the pain |
| ✅ Good | Proper OOD — `Vehicle`, `Spot`, `Level`, `ParkingLot`, `Ticket` | Separation of concerns, easier to read/test |
| 🌟 Best | Good + **Strategy pattern** for pricing | Open/closed: add new pricing plans without edits |


## 🛠️ Setup

```bash
cd 07-object-oriented-design/parking-lot
uv sync
```

Select the `.venv` kernel in VS Code. If not visible, reload: `Cmd+Shift+P` → **Reload Window**.


## 🚫 Version 1 — The "God class" (anti-pattern)

All logic — storing spots, tracking vehicles, computing fees — sits on one class.
It *works* for small demos but is painful to maintain.


In [ ]:
import time

class BadParkingLot:
    """Everything crammed into one class. Don't do this in real code."""
    def __init__(self, num_motorcycle, num_car, num_truck):
        # parallel lists = easy to get out of sync
        self.spots_moto  = [None] * num_motorcycle
        self.spots_car   = [None] * num_car
        self.spots_truck = [None] * num_truck
        self.entry_times = {}   # plate -> entry_time
        self.parked_at   = {}   # plate -> (bucket_name, index)

    def park(self, plate, size):
        # giant if/else ladder — the classic smell
        if size == 'motorcycle':
            buckets = [('moto', self.spots_moto), ('car', self.spots_car), ('truck', self.spots_truck)]
        elif size == 'car':
            buckets = [('car', self.spots_car), ('truck', self.spots_truck)]
        elif size == 'truck':
            buckets = [('truck', self.spots_truck)]
        else:
            raise ValueError(f'unknown size {size}')

        for name, bucket in buckets:
            for i, occupant in enumerate(bucket):
                if occupant is None:
                    bucket[i] = plate
                    self.entry_times[plate] = time.time()
                    self.parked_at[plate]   = (name, i)
                    return f'parked {plate} in {name} spot #{i}'
        raise RuntimeError('full')

    def leave(self, plate, now=None):
        if plate not in self.parked_at:
            raise KeyError(plate)
        name, i = self.parked_at.pop(plate)
        entry  = self.entry_times.pop(plate)
        bucket = {'moto': self.spots_moto, 'car': self.spots_car, 'truck': self.spots_truck}[name]
        bucket[i] = None
        hours = max(1, ((now or time.time()) - entry) / 3600)
        # pricing logic hard-coded here too
        rate  = {'moto': 2, 'car': 2, 'truck': 4}
        # Hidden bug: fee is keyed on the SPOT bucket, not the vehicle's real size.
        # A motorcycle that overflowed into a car spot gets billed as a car.
        fee = rate[name] * hours
        return round(fee, 2)


bad = BadParkingLot(num_motorcycle=1, num_car=2, num_truck=1)
print(bad.park('BIKE-1', 'motorcycle'))
print(bad.park('CAR-1',  'car'))
t0  = time.time()
fee = bad.leave('CAR-1', now=t0 + 7200)   # 2h later
print('car paid $', fee)


### What's wrong with Version 1?

- **No clear entities.** "A spot" is just an index into a list — you can't attach behavior to it.
- **Parallel dictionaries** (`entry_times`, `parked_at`) can drift out of sync.
- **`if/else` ladder** on vehicle type → every new vehicle size means editing this method
  (violates the **O**pen/Closed principle).
- **Hidden bug:** notice the comment — fee uses the *spot bucket name*, not the vehicle's true size.
  A motorcycle that overflowed into a car spot would be charged as a car.
  This is exactly the kind of bug bad data modeling hides.
- **Pricing is hard-coded** inside `leave` — can't swap in a monthly-pass rule without surgery.


## ✅ Version 2 — Proper OOD

We split responsibilities into small classes, each doing one thing well.


In [ ]:
from enum import Enum
from dataclasses import dataclass, field
import itertools, time

class Size(Enum):
    MOTORCYCLE = 1
    CAR        = 2
    TRUCK      = 3

@dataclass
class Vehicle:
    plate: str
    size: Size

class Spot:
    _ids = itertools.count(1)
    def __init__(self, size: Size):
        self.id      = next(Spot._ids)
        self.size    = size
        self.vehicle = None                                  # None == free

    def can_fit(self, v: Vehicle) -> bool:
        return self.vehicle is None and v.size.value <= self.size.value

    # A Spot is the guardian of its own invariant: "at most one vehicle, and only
    # one that physically fits". Callers must not be able to corrupt it, even by
    # mistake -- so park()/leave() raise instead of silently overwriting.
    def park(self, v: Vehicle):
        if not self.can_fit(v):
            raise ValueError(f'{v.plate} cannot park in {self!r}')
        self.vehicle = v

    def leave(self):
        if self.vehicle is None:
            raise ValueError(f'{self!r} is already free')
        self.vehicle = None

    def __repr__(self):
        who = 'free' if self.vehicle is None else self.vehicle.plate
        return f'Spot#{self.id}({self.size.name},{who})'

class Level:
    def __init__(self, floor: int, spots):
        self.floor, self.spots = floor, spots

    def find_spot(self, v: Vehicle):
        # simple first-fit; could be smarter (closest-fit) -- see extensions
        return next((s for s in self.spots if s.can_fit(v)), None)

@dataclass
class Ticket:
    vehicle:    Vehicle
    spot:       Spot
    entry_time: float = field(default_factory=time.time)
    closed:     bool  = False    # a ticket is single-use: pay once, leave once

class ParkingLot:
    RATE_PER_HOUR = {Size.MOTORCYCLE: 1, Size.CAR: 2, Size.TRUCK: 4}

    def __init__(self, levels):
        self.levels = levels

    def park(self, v: Vehicle) -> Ticket:
        for lvl in self.levels:
            spot = lvl.find_spot(v)
            if spot:
                spot.park(v)
                return Ticket(v, spot)
        raise RuntimeError('lot full')

    # Template method: the *invariant* half (single-use ticket, release the spot)
    # is fixed here; only the *policy* half (_fee) is meant to be overridden.
    def leave(self, t: Ticket, now=None) -> float:
        if t.closed:
            raise ValueError('ticket already used -- would double-charge the driver')
        fee = self._fee(t, now or time.time())
        t.spot.leave()
        t.closed = True
        return fee

    def _fee(self, t: Ticket, exit_time: float) -> float:
        hours = max(1, (exit_time - t.entry_time) / 3600)
        # pricing keyed on the VEHICLE's size, not the spot's -- bug from v1 fixed
        return round(self.RATE_PER_HOUR[t.vehicle.size] * hours, 2)

    def available(self):
        out = {s: 0 for s in Size}
        for lvl in self.levels:
            for s in lvl.spots:
                if s.vehicle is None:
                    out[s.size] += 1
        return out

In [ ]:
# Walk-through
lot = ParkingLot([
    Level(1, [Spot(Size.MOTORCYCLE), Spot(Size.CAR), Spot(Size.CAR), Spot(Size.TRUCK)]),
    Level(2, [Spot(Size.CAR), Spot(Size.CAR)]),
])

t1 = lot.park(Vehicle('BIKE-1',  Size.MOTORCYCLE))
t2 = lot.park(Vehicle('CAR-1',   Size.CAR))
t3 = lot.park(Vehicle('TRUCK-1', Size.TRUCK))

for lvl in lot.levels:
    print(f'level {lvl.floor}:', lvl.spots)

# Charge the car for a 2-hour stay
fee = lot.leave(t2, now=t2.entry_time + 7200)
print(f'CAR-1 paid ${fee}')
print('available after car left:', lot.available())


In [ ]:
# Fit-rule tests — the bug from v1 no longer exists
lot2 = ParkingLot([Level(1, [Spot(Size.CAR), Spot(Size.MOTORCYCLE)])])

# Trucks don't fit in car spots
try:
    lot2.park(Vehicle('BIG', Size.TRUCK))
except RuntimeError as e:
    print('expected error:', e)

# A motorcycle can overflow into a car spot — and is still charged bike-rate
t = lot2.park(Vehicle('MOTO', Size.MOTORCYCLE))
print('parked:', t.spot)
fee = lot2.leave(t, now=t.entry_time + 7200)
print(f'bike paid ${fee}  (correctly billed as motorcycle, not car)')


### Why Version 2 is better

- **Each class has one job** (Single Responsibility).
- Adding a new vehicle size means: add an enum entry + add `Spot`s of that size.
  No `if/else` ladders to maintain (Open/Closed).
- `Ticket` is a **value object** -- easy to pass around and test.
- Bug fixed: pricing is keyed on the vehicle's size, not the spot's.

#### 🔒 Who enforces the invariants?

An entity is only trustworthy if it *cannot* be put into an illegal state, even by
buggy caller code. Notice the two guards we added:

| Invariant | Enforced by | What it stops |
|---|---|---|
| A spot holds **at most one** vehicle, and only one that fits | `Spot.park()` raises | A second `park()` silently overwriting the first car -- the car "disappears" |
| A spot can only be freed if it was occupied | `Spot.leave()` raises | Phantom frees that inflate `available()` |
| A ticket is **single-use** | `ParkingLot.leave()` checks `t.closed` | Charging the same driver twice / freeing a spot someone else now occupies |

> 🧠 **Interview line:** *"I put the check in `Spot.park`, not in `ParkingLot.park`,
> because the spot owns that invariant. If I ever add a second caller -- a valet
> service, an admin tool -- it gets the protection for free."*
> That is encapsulation doing real work, not just making fields private.

But pricing is still hard-coded in `ParkingLot`. What if we need:

- a monthly-pass holder who pays $0 at exit?
- a weekend promotion at 50% off?
- surge pricing when the lot is >80% full?

We'd end up editing `leave()` every time -- back to the Open/Closed violation. Let's fix that next.

## 🌟 Version 3 -- Best: Strategy pattern for pricing

The **Strategy pattern** says: *"Instead of hard-coding one algorithm, take the
algorithm as a parameter."* We define a `PricingStrategy` interface and let the
`ParkingLot` use whichever strategy it's given.

> ⚠️ **Where do you plug the strategy in?** Not by overriding `leave()`. `leave()`
> also enforces the single-use-ticket rule and releases the spot -- an override that
> forgets those re-introduces the double-charge bug. Instead `ParkingLot.leave()` is a
> **template method**: it owns the invariants and calls `_fee()` for the part that varies.
> Subclasses override `_fee()` only. Same idea as `Vehicle.size`: put the thing that
> changes behind a seam, and keep the thing that must never change out of reach.

In [ ]:
from abc import ABC, abstractmethod
import math

class PricingStrategy(ABC):
    """Contract: given a ticket and an exit time, return the fee as a float.

    The *return type* is part of the interface. If one strategy returns an int
    and another a float, every caller has to defend itself -- so each concrete
    strategy coerces to float rather than leaving it to chance.
    """
    @abstractmethod
    def fee(self, ticket: Ticket, exit_time: float) -> float: ...

class HourlyPricing(PricingStrategy):
    def __init__(self, rates):
        self.rates = rates
    def fee(self, ticket, exit_time):
        hours = max(1, (exit_time - ticket.entry_time) / 3600)
        return float(round(self.rates[ticket.vehicle.size] * hours, 2))

class FlatDailyPricing(PricingStrategy):
    def __init__(self, price_per_day):
        self.price_per_day = price_per_day
    def fee(self, ticket, exit_time):
        days = math.ceil((exit_time - ticket.entry_time) / 86400) or 1
        return float(round(self.price_per_day * days, 2))

class MonthlyPassPricing(PricingStrategy):
    """Pass-holders don't pay at exit; they paid upfront for the month."""
    def fee(self, ticket, exit_time):
        return 0.0

class BetterParkingLot(ParkingLot):
    def __init__(self, levels, pricing: PricingStrategy):
        super().__init__(levels)
        self.pricing = pricing

    # We override ONLY _fee. `leave()` -- and therefore the single-use-ticket
    # guard and the spot release -- is inherited untouched. Overriding `leave()`
    # itself would silently drop those invariants: a classic inheritance trap.
    def _fee(self, t: Ticket, exit_time: float) -> float:
        return self.pricing.fee(t, exit_time)


In [ ]:
# Demo: same lot, three different pricing plans
def make_lot(pricing):
    return BetterParkingLot(
        [Level(1, [Spot(Size.CAR), Spot(Size.TRUCK)])],
        pricing=pricing,
    )

hourly  = make_lot(HourlyPricing({Size.MOTORCYCLE: 1, Size.CAR: 2, Size.TRUCK: 4}))
daily   = make_lot(FlatDailyPricing(price_per_day=20))
monthly = make_lot(MonthlyPassPricing())

for name, lot in [('hourly', hourly), ('daily', daily), ('monthly-pass', monthly)]:
    car = Vehicle('CAR-42', Size.CAR)   # fresh vehicle per lot
    t   = lot.park(car)
    fee = lot.leave(t, now=t.entry_time + 7200)    # 2 hours
    print(f'{name:>12s} plan -> ${fee}')


### Why Version 3 is the "best" version

- **Open/closed.** Adding a new pricing plan (surge, promo, loyalty) = one new class.
  We **never touch** `BetterParkingLot` again.
- **Dependency inversion.** `BetterParkingLot` depends on the `PricingStrategy`
  *abstraction*, not on concrete pricing rules.
- **Unit testing is trivial** — each strategy is tested in isolation.
- **Composable.** You can imagine a `DiscountedPricing(wrapped, percent)` decorator
  that takes any other strategy and applies a percentage off.

> 🧠 **Takeaway:** `if/else` ladder in V1 → subclasses in V2 → pluggable strategy in V3.
> That is the "bad → good → best" arc that interviewers love to see.


### 🧪 Mini-tests (assertions)

Quick sanity checks — great habit for design interviews too.

In [ ]:
import contextlib

def must_raise(exc, fn, *a, **kw):
    """Assert that fn(*a) raises `exc`. Fails loudly if it does NOT raise --
    a bare try/except would silently 'pass' when the guard is missing."""
    try:
        fn(*a, **kw)
    except exc:
        return True
    raise AssertionError(f'expected {exc.__name__} from {getattr(fn, "__name__", fn)}')

lot = BetterParkingLot(
    [Level(1, [Spot(Size.MOTORCYCLE), Spot(Size.CAR), Spot(Size.TRUCK)])],
    pricing=HourlyPricing({Size.MOTORCYCLE: 1, Size.CAR: 2, Size.TRUCK: 4}),
)

# 1. Parking returns a ticket whose spot remembers the vehicle
t = lot.park(Vehicle('T-1', Size.CAR))
assert t.spot.vehicle.plate == 'T-1', 'spot should remember its vehicle'

# 2. Leaving frees the spot and charges correctly
fee = lot.leave(t, now=t.entry_time + 3600)   # exactly 1 hour
assert fee == 2.0, f'expected $2 for 1h car, got ${fee}'
assert t.spot.vehicle is None, 'spot should be free after leave'

# 3. A ticket is single-use -- no double charge, no double free
must_raise(ValueError, lot.leave, t, now=t.entry_time + 7200)

# 4. Lot full -> RuntimeError (note: must_raise FAILS if nothing is raised)
tiny = BetterParkingLot([Level(1, [Spot(Size.CAR)])], HourlyPricing({Size.CAR: 2}))
tiny.park(Vehicle('A', Size.CAR))
must_raise(RuntimeError, tiny.park, Vehicle('B', Size.CAR))

# 5. Spot invariants: no overwriting an occupant, no fitting an oversized vehicle
occupied = Spot(Size.CAR); occupied.park(Vehicle('X', Size.CAR))
must_raise(ValueError, occupied.park, Vehicle('Y', Size.CAR))
must_raise(ValueError, Spot(Size.CAR).park, Vehicle('BIG', Size.TRUCK))
must_raise(ValueError, Spot(Size.CAR).leave)          # freeing an already-free spot

# 6. Strategy contract: every strategy answers fee() for the same ticket
tk = Ticket(Vehicle('S-1', Size.CAR), Spot(Size.CAR))
for strat in (HourlyPricing({Size.CAR: 2}), FlatDailyPricing(20), MonthlyPassPricing()):
    assert isinstance(strat.fee(tk, tk.entry_time + 3600), float)

print('all assertions passed ✅')

👉 **Next:** Notebook 3 adds real-world extensions — thread-safe parking
(multiple entrances), reserved / EV spots, and pluggable payment providers.